In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,roc_auc_score, confusion_matrix,classification_report)

In [2]:
df = pd.read_csv(r'C:\Infoct_project 1 sample practice\dataset\predictive_maintenance.csv')
print("Dataset loaded!")
print(df.shape)

Dataset loaded!
(10000, 10)


In [3]:
# Week 2 external context
np.random.seed(42)
df['Ambient_temperature'] = np.random.normal(loc=295, scale=3, size=len(df))
df['Load_density'] = np.random.uniform(low=0.3, high=1.0, size=len(df))

# Week 1 engineered features
df["temp_difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]
df["tool_wear_rate"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"]
df["heat_stress_index"] = df["Air temperature [K]"] * df["Tool wear [min]"]
df["wear_per_rotation"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"] * 1000

# Week 3 new features
df["thermal_stress"] = df["temp_difference"] * df["tool_wear_rate"]
df["power_per_temp"] = df["power"] / df["Air temperature [K]"]

print("All features recreated!")
print(df.shape)

All features recreated!
(10000, 19)


In [4]:
# Final 12 features from Week 3
final_features = ['Air temperature [K]', 'Process temperature [K]','Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temp_difference', 'power', 'tool_wear_rate',
'heat_stress_index', 'wear_per_rotation','thermal_stress', 'power_per_temp']

X = df[final_features]
Y = df['Target']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train LightGBM
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train_scaled, Y_train)
print("Model trained successfully!")

[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Model trained successfully!


In [5]:
# Evaluation framework function
def evaluate_model(y_true, y_pred, y_proba, dataset_name="Dataset"):
    print(f"\n=== Evaluation Report: {dataset_name} ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='macro'):.3f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='macro'):.3f}")
    print(f"ROC-AUC:   {roc_auc_score(y_true, y_proba):.3f}")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred))

# Test on original clean data
pred_original = model.predict(X_test_scaled)
proba_original = model.predict_proba(X_test_scaled)[:, 1]

evaluate_model(Y_test, pred_original, proba_original, "Original Clean Data")


=== Evaluation Report: Original Clean Data ===
Accuracy:  0.987
Precision: 0.891
Recall:    0.908
F1 Score:  0.899
ROC-AUC:   0.973

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.79      0.82      0.81        68

    accuracy                           0.99      2000
   macro avg       0.89      0.91      0.90      2000
weighted avg       0.99      0.99      0.99      2000



c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
# Save baseline results
baseline_results = {
    'dataset': 'Original Clean Data',
    'noise_level': '0%',
    'accuracy': accuracy_score(Y_test, pred_original),
    'precision': precision_score(Y_test, pred_original, average='macro'),
    'recall': recall_score(Y_test, pred_original, average='macro'),
    'f1_score': f1_score(Y_test, pred_original, average='macro'),
    'roc_auc': roc_auc_score(Y_test, proba_original)
}

baseline_df = pd.DataFrame([baseline_results])
print("=== Baseline Results Saved ===")
print(baseline_df)

# Save to CSV
baseline_df.to_csv(
    r'C:\Infoct_project 1 sample practice\Week4_Yogesh\baseline_results.csv',
    index=False)
print("\nBaseline results saved to CSV!")

=== Baseline Results Saved ===
               dataset noise_level  accuracy  precision    recall  f1_score  \
0  Original Clean Data          0%    0.9865   0.891256  0.907883  0.899381   

    roc_auc  
0  0.973009  

Baseline results saved to CSV!
